# Homework 4.  Implementing a FASTA file parser and calculating FASTA statistics using command line tools from Python

# Class

In [1]:
import requests
import re
from Bio import SeqIO
import subprocess

In [2]:
#mini 'kostyl', I'm sorry :)
#create conda env, install seqkit and run .ipynb in this env
path_to_cli_seqkit = '/home/polinam/miniconda3/envs/seqkit/bin/seqkit'

In [3]:
def http_function(endpoint, **headers):
    response = requests.get(endpoint, headers=headers)
    return response.json() if response.status_code == 200 else {'error': response.text}

class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    def _get_uniprot(self, accession):
        endpoint = f'https://rest.uniprot.org/uniprotkb/{accession}.json'
        return http_function(endpoint)

    def _get_ensembl(self, id):
        endpoint = f'https://rest.ensembl.org/lookup/id/{id}'
        headers = {'Content-Type': 'application/json'}
        return http_function(endpoint, **headers)

    def _uniprot_parse_response(self, resp: dict):
        if 'messages' in resp or 'error' in resp:
            return {'error': resp.get('messages', resp.get('error'))}
        output = {
            resp.get('primaryAccession'): {
                'organism': resp.get('organism', {}).get('scientificName'),
                'geneInfo': resp.get('genes'),
                'sequenceInfo': resp.get('sequence'),
                'type': 'protein'
            }
        }
        return output

    def _ensembl_parse_response(self, resp: dict):
        if 'error' in resp:
            return {'error': resp.get('error')}
        keys = ['object_type', 'assembly_name', 'species', 'db_type', 'biotype', 'display_name',
                'id', 'description', 'canonical_transcript', 'source']
        output = {resp.get('id'): {k: resp.get(k) for k in keys}}
        return output

    def _access_database(self, id, database, seq_description, seq_sequence):
        if database.lower() == 'uniprot':
            resp = self._get_uniprot(id)
            parsed = self._uniprot_parse_response(resp)
        else:
            resp = self._get_ensembl(id)
            parsed = self._ensembl_parse_response(resp)

        return {
            'description': seq_description,
            'sequence': seq_sequence,
            'db': database,
            'db_info': parsed
        }

    def seqkit_stats(self):
        seqkit_path = path_to_cli_seqkit
        try:
            result = subprocess.run(
                [seqkit_path, 'stats', '--all', '--tabular', self.filename],
                capture_output=True, text=True, check=True
            )
            lines = result.stdout.strip().split('\n')
            if len(lines) < 2:
                return {'error': 'No stats'}

            header = lines[0].split()
            values = lines[1].split()
            stat_dict = {k: v for k, v in zip(header, values)}
            return {
                'fasta_seqkit_stat_info': stat_dict,
                'fasta_type': stat_dict.get('type', 'Unknown'),
                'fasta_num_seqs': int(stat_dict.get('num_seqs', 0))
            }

        except subprocess.CalledProcessError as e:
            return {'error': e.stderr}
        except FileNotFoundError:
            return {'error': 'Seqkit not found'}

    def biopython_parser(self, seqkit_result):
        file_type = seqkit_result.get('fasta_type', 'Protein')
        if file_type == 'Protein':
            database = 'uniprot'
            id_pattern = re.compile(r'\b[A-Z0-9]{1,6}(?:_[0-9]+)?\b')
        elif file_type == 'DNA':
            database = 'ensembl'
            id_pattern = re.compile(r'ENS[A-Z]*[GTEP][0-9]{11}')
        else:
            database = None
            id_pattern = None

        output = {'DB_name': database}
        warnings = set()

        for record in SeqIO.parse(self.filename, 'fasta'):
            desc = record.description
            seq = str(record.seq)
            seq_ids = id_pattern.findall(desc) if id_pattern else []

            if not seq_ids:
                key = f'file_info_{record.id}'
                output[key] = {
                    'description': desc,
                    'sequence': seq,
                    f'database_info_None': None
                }
                warnings.add('No ID match found')
                continue

            for seq_id in seq_ids:
                db_info = self._access_database(seq_id, database, desc, seq)
                file_key = f'file_info_{seq_id}'
                output[file_key] = {
                    'description': db_info['description'],
                    'sequence': db_info['sequence'],
                    f'database_info_{seq_id}': db_info['db_info']
                }

        if warnings:
            output['WARNING'] = warnings

        return output

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

# test_file.fasta

In [4]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
stats

{'fasta_seqkit_stat_info': {'file': 'test_file.fasta',
  'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': '2',
  'sum_len': '456',
  'min_len': '29',
  'avg_len': '228.0',
  'max_len': '427',
  'Q1': '29',
  'Q2': '228',
  'Q3': '427',
  'sum_gap': '0',
  'N50': '427',
  'N50_num': '1',
  'Q20(%)': '0',
  'Q30(%)': '0',
  'AvgQual': '0.00',
  'GC(%)': '0.00',
  'sum_n': '0'},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 2}

In [5]:
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
	database_info_P11473
		P11473
			organism
				Homo sapiens
			geneInfo
				[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
			sequenceInfo
				value
					MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRP

# ensembl_download_1.fasta

In [6]:
parser_1 = MyFastaParser('ensembl_download_1.fasta')
stats_1 = parser_1.seqkit_stats()

stats_1

{'fasta_seqkit_stat_info': {'file': 'ensembl_download_1.fasta',
  'format': 'FASTA',
  'type': 'DNA',
  'num_seqs': '6',
  'sum_len': '86',
  'min_len': '9',
  'avg_len': '14.3',
  'max_len': '23',
  'Q1': '10',
  'Q2': '14',
  'Q3': '17',
  'sum_gap': '0',
  'N50': '16',
  'N50_num': '3',
  'Q20(%)': '0',
  'Q30(%)': '0',
  'AvgQual': '0.00',
  'GC(%)': '45.35',
  'sum_n': '0'},
 'fasta_type': 'DNA',
 'fasta_num_seqs': 6}

In [7]:
biopython_1 = parser_1.biopython_parser(stats_1)
parser_1.show_output(biopython_1)

DB_name
	ensembl
file_info_ENSMUST00000196221
	description
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
	database_info_ENSMUST00000196221
		ENSMUST00000196221
			object_type
				Transcript
			assembly_name
				GRCm39
			species
				mus_musculus
			db_type
				core
			biotype
				TR_D_gene
			display_name
				Trdd1-202
			id
				ENSMUST00000196221
			description
				None
			canonical_transcript
				None
			source
				havana
file_info_ENSMUSG00000096749
	description
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
	database_info_ENSMUSG00000096749
		ENSM

# ensembl_download_2.fasta

In [8]:
parser_2 = MyFastaParser('ensembl_download_2.fasta')
stats_2 = parser_2.seqkit_stats()

stats_2

{'fasta_seqkit_stat_info': {'file': 'ensembl_download_2.fasta',
  'format': 'FASTA',
  'type': 'DNA',
  'num_seqs': '6',
  'sum_len': '463',
  'min_len': '11',
  'avg_len': '77.2',
  'max_len': '350',
  'Q1': '12',
  'Q2': '15',
  'Q3': '60',
  'sum_gap': '0',
  'N50': '350',
  'N50_num': '1',
  'Q20(%)': '0',
  'Q30(%)': '0',
  'AvgQual': '0.00',
  'GC(%)': '43.84',
  'sum_n': '0'},
 'fasta_type': 'DNA',
 'fasta_num_seqs': 6}

In [9]:
biopython_2 = parser_2.biopython_parser(stats_2)
parser_2.show_output(biopython_2)

DB_name
	ensembl
file_info_ENSDART00000165410
	description
		ENSDART00000165410.2 cds chromosome:GRCz11:2:31869121:31869554:-1 gene:ENSDARG00000100191.2 gene_biotype:TR_V_gene transcript_biotype:TR_V_gene gene_symbol:trgv6 description:T cell receptor gamma variable 6 [Source:ZFIN;Acc:ZDB-GENE-051115-10]
	sequence
		ATGAGTCTTCAAATGCTCTTCTGCTTTTTCTTTCTTTTTAACTTTTATGCAGTTGAAGGACAAGTGACGCTGCGACAGAAAATATCCTCAACCAAATCTCAGGACAAGACTGTTGTCATAGACTGTGATTACCCTTCAGACTGCCGCAGCTACATTCACTGGTACCAACTAAAAGGACAAACCTTAAAGAGAATATTATATGCACAAATTTCAGGAGGAGAACCAGCCAAAGATGCTGGCTTTGAGTTGTTTAAAATAGACCGTAAACAGTCAAATATTGCTCTGAAAATACCTGAACTGAAAACAGAGCATTCAGCAGTCTATTACTGTGCTTGTTGGGTCTCGGGCGG
	database_info_ENSDART00000165410
		ENSDART00000165410
			object_type
				Transcript
			assembly_name
				GRCz11
			species
				danio_rerio
			db_type
				core
			biotype
				TR_V_gene
			display_name
				trgv6-201
			id
				ENSDART00000165410
			description
				None
			canonical_transcript
				None
			source
				havana
file_info_

# uniprot_download.fasta

In [10]:
parser_3 = MyFastaParser('uniprot_download.fasta')
stats_3 = parser_3.seqkit_stats()

stats_3

{'fasta_seqkit_stat_info': {'file': 'uniprot_download.fasta',
  'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': '7',
  'sum_len': '3861',
  'min_len': '180',
  'avg_len': '551.6',
  'max_len': '1382',
  'Q1': '429',
  'Q2': '441',
  'Q3': '500',
  'sum_gap': '0',
  'N50': '468',
  'N50_num': '3',
  'Q20(%)': '0',
  'Q30(%)': '0',
  'AvgQual': '0.00',
  'GC(%)': '0.00',
  'sum_n': '0'},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 7}

In [11]:
biopython_3 = parser_3.biopython_parser(stats_3)
parser_3.show_output(biopython_3)

DB_name
	uniprot
file_info_Q9R1A7
	description
		sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
	sequence
		MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQLLRIQDTHPFATPLMQELFSSTDG
	database_info_Q9R1A7
		Q9R1A7
			organism
				Rattus norvegicus
			geneInfo
				[{'geneName': {'value': 'Nr1i2'}, 'synonyms': [{'value': 'Pxr'}]}]
			sequenceInfo
				value
					MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLR